# Ingestão Silver: Bronze ➔ Silver com regras de Data Quality
Este notebook realiza, para cada micro-lote (arquivo) ainda não processado da Bronze:
- Leitura da `bronze_ecommerce_enderecos` filtrada por `bronze_source_file`.
- Aplicação das 10 regras de qualidade da tabela `ecommerce_enderecos` (Squad 1).
- Uma coluna booleana por regra: **`True` = a linha FALHOU na regra**, `False` = passou.
- Coluna `silver_processed_at` e carga Delta (`append`) na `silver_ecommerce_enderecos`.

| # | Coluna | Regra |
|---|--------|-------|
| 1 | `falha_r01_id_endereco_nulo_ou_duplicado` | `id_endereco` não pode ser nulo nem duplicado |
| 2 | `falha_r02_id_cliente_invalido` | `id_cliente` deve existir em `ecommerce_clientes` (FK) |
| 3 | `falha_r03_cep_invalido` | `cep` com exatamente 8 dígitos numéricos |
| 4 | `falha_r04_uf_invalida` | `estado` deve ser uma das 26 UFs + DF |
| 5 | `falha_r05_coordenadas_invalidas` | lat/long não nulas e dentro do Brasil |
| 6 | `falha_r06_principal_invalido` | exatamente 1 `is_principal = TRUE` por cliente |
| 7 | `falha_r07_apelido_invalido` | `apelido` em `Casa`, `Trabalho`, `Outro` |
| 8 | `falha_r08_mais_de_3_enderecos` | no máximo 3 endereços por cliente |
| 9 | `falha_r09_logradouro_vazio` | `logradouro` não nulo nem vazio |
| 10 | `falha_r10_numero_nulo` | `numero` não pode ser nulo |

In [ ]:
import time
from pyspark.sql import Window
from pyspark.sql import functions as F

# ==============================================================================
# 1. Parâmetros
# ==============================================================================
bronze_table = "bronze_ecommerce_enderecos"
silver_table = "silver_ecommerce_enderecos"
# Tabela de referência para a FK da regra 2 (ajustar se o nome for outro)
clientes_ref_table = "bronze_ecommerce_clientes"
poll_interval_seconds = 15

UFS_VALIDAS = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA", "MT", "MS", "MG", "PA", "PB",
    "PR", "PE", "PI", "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO",
]
APELIDOS_VALIDOS = ["Casa", "Trabalho", "Outro"]
LAT_MIN, LAT_MAX = -33.75, 5.27
LON_MIN, LON_MAX = -73.99, -32.39
MAX_ENDERECOS_POR_CLIENTE = 3

In [ ]:
# ==============================================================================
# 2. Regras de Data Quality (True = linha FALHOU na regra)
# ==============================================================================
def aplicar_regras_qualidade(df):
    """Recebe o micro-lote da Bronze e devolve o DataFrame com as 10 colunas de falha."""
    w_id      = Window.partitionBy("id_endereco")
    w_cliente = Window.partitionBy("id_cliente")

    # Regra 1: duplicidade dentro do micro-lote e contra ids já existentes na Silver
    df = df.withColumn("_qtd_id_lote", F.count("*").over(w_id))
    if spark.catalog.tableExists(silver_table):
        ids_silver = spark.table(silver_table).select("id_endereco").distinct().withColumn("_ja_na_silver", F.lit(True))
        df = df.join(ids_silver, on="id_endereco", how="left")
    else:
        df = df.withColumn("_ja_na_silver", F.lit(None).cast("boolean"))

    # Regra 2: FK em ecommerce_clientes (se a tabela de referência ainda não existir,
    # valida apenas que id_cliente não é nulo)
    if spark.catalog.tableExists(clientes_ref_table):
        clientes = spark.table(clientes_ref_table).select("id_cliente").distinct().withColumn("_cliente_existe", F.lit(True))
        df = df.join(clientes, on="id_cliente", how="left")
    else:
        print(f"⚠ '{clientes_ref_table}' não encontrada: regra 2 valida somente id_cliente não nulo.")
        df = df.withColumn("_cliente_existe", F.when(F.col("id_cliente").isNotNull(), F.lit(True)))

    return (
        df
        .withColumn("falha_r01_id_endereco_nulo_ou_duplicado",
                    F.col("id_endereco").isNull() | (F.col("_qtd_id_lote") > 1) | F.coalesce(F.col("_ja_na_silver"), F.lit(False)))
        .withColumn("falha_r02_id_cliente_invalido",
                    F.col("id_cliente").isNull() | ~F.coalesce(F.col("_cliente_existe"), F.lit(False)))
        .withColumn("falha_r03_cep_invalido",
                    F.col("cep").isNull() | ~F.col("cep").cast("string").rlike(r"^[0-9]{8}$"))
        .withColumn("falha_r04_uf_invalida",
                    F.col("estado").isNull() | ~F.upper(F.trim(F.col("estado"))).isin(UFS_VALIDAS))
        .withColumn("falha_r05_coordenadas_invalidas",
                    F.col("latitude").isNull() | F.col("longitude").isNull()
                    | ~F.col("latitude").between(LAT_MIN, LAT_MAX)
                    | ~F.col("longitude").between(LON_MIN, LON_MAX))
        .withColumn("falha_r06_principal_invalido",
                    F.sum(F.when(F.col("is_principal") == True, 1).otherwise(0)).over(w_cliente) != 1)
        .withColumn("falha_r07_apelido_invalido",
                    F.col("apelido").isNull() | ~F.col("apelido").isin(APELIDOS_VALIDOS))
        .withColumn("falha_r08_mais_de_3_enderecos",
                    F.count("*").over(w_cliente) > MAX_ENDERECOS_POR_CLIENTE)
        .withColumn("falha_r09_logradouro_vazio",
                    F.col("logradouro").isNull() | (F.trim(F.col("logradouro")) == ""))
        .withColumn("falha_r10_numero_nulo", F.col("numero").isNull())
        .drop("_qtd_id_lote", "_ja_na_silver", "_cliente_existe")
    )

In [ ]:
# ==============================================================================
# 3. Loop de Monitoramento: micro-lote Silver
# ==============================================================================
def arquivos_pendentes():
    """Arquivos presentes na Bronze que ainda não foram processados na Silver."""
    if not spark.catalog.tableExists(bronze_table):
        return []
    bronze_files = {r["bronze_source_file"] for r in spark.table(bronze_table).select("bronze_source_file").distinct().collect()}
    silver_files = set()
    if spark.catalog.tableExists(silver_table):
        silver_files = {r["bronze_source_file"] for r in spark.table(silver_table).select("bronze_source_file").distinct().collect()}
    return sorted(bronze_files - silver_files)


print(f"Iniciando monitoramento da Bronze '{bronze_table}'...\n")

while True:
    try:
        for source_file in arquivos_pendentes():
            print(f"Processando micro-lote: {source_file}")

            df_lote = (
                spark.table(bronze_table)
                .filter(F.col("bronze_source_file") == source_file)
                .drop("ano", "mes", "dia", "hora")
            )

            df_silver = aplicar_regras_qualidade(df_lote).withColumn("silver_processed_at", F.current_timestamp())

            df_silver.write.format("delta").mode("append").saveAsTable(silver_table)

            regras = [c for c in df_silver.columns if c.startswith("falha_")]
            resumo = df_silver.agg(F.count("*").alias("total"), *[F.sum(F.col(c).cast("int")).alias(c) for c in regras]).collect()[0]
            print(f"✔ '{source_file}' gravado na Silver: {resumo['total']} registros.")
            for c in regras:
                print(f"   {c}: {resumo[c]} falha(s)")

        time.sleep(poll_interval_seconds)

    except KeyboardInterrupt:
        print("\nProcesso interrompido manualmente pelo usuário.")
        break
    except Exception as e:
        print(f"\n Erro durante a execução do loop: {e}")
        time.sleep(poll_interval_seconds)